In [ ]:
mots_polysemique=["café", "souris","langue","fraise","avocat", "bois", "livre", "vol", "action", 'tour']
mots_monosemique=["médecin", "volcan", "pentathlon", "atome", "photosynthèse","chrysanthème","hydrogène","électron","antibiotique","alambic"]
mots_cible= mots_polysemique + mots_monosemique 

In [ ]:
import wikipediaapi
import re
import pandas as pd
wiki = wikipediaapi.Wikipedia(
    language='fr',
    user_agent='projet-polysemie/1.0'
)

In [ ]:
def extraire_phrases(texte, mot, n_max=30):
    """Extrait les phrases contenant le mot cible."""
    # Découpe en phrases
    phrases = re.split(r'(?<=[.!?])\s+', texte)
    
    # Garde seulement celles qui contiennent le mot (insensible à la casse)
    pattern = r'\b' + re.escape(mot) + r'\b'
    resultats = [p.strip() for p in phrases 
                 if re.search(pattern, p, re.IGNORECASE) 
                 and 20 < len(p) < 300]  # phrases ni trop courtes ni trop longues
    
    return resultats[:n_max]

articles_par_mot = {
    # -------- POLYSÉMIQUES --------
    "café":    ["Café", "Café (boisson)", "Café (établissement)"],
    "souris":  ["Souris", "Souris (informatique)"],
    "langue":  ["Langue", "Langue (anatomie)", "Langue vivante", "Linguistique"],
    "fraise":  ["Fraise", "Fraise (outil)", "Fraise (costume)"],
    "avocat":  ["Avocat (fruit)", "Avocat (métier)"],
    "bois":    ["Bois", "Bois (matériau)", "Bois (anatomie)"],
    "livre":   ["Livre (document)", "Livre (unité de masse)", "Livre sterling"],
    "vol":     ["Vol (aviation)", "Vol (droit)", "Migration des oiseaux"],
    "action":  ["Action (finance)", "Action (philosophie)", "Action (droit)", "Cinéma"],
    "tour":    ["Tour (architecture)", "Tour (machine-outil)",
                "Tour de France", "Tour de magie", "Tournée"],

    # -------- MONOSÉMIQUES --------
    "médecin":      ["Médecin"],
    "volcan":       ["Volcan", "Éruption volcanique"],
    "pentathlon":   ["Pentathlon", "Pentathlon moderne"],
    "atome":        ["Atome"],
    "photosynthèse":["Photosynthèse"],
    "chrysanthème": ["Chrysanthème"],
    "hydrogène":    ["Hydrogène"],
    "électron":     ["Électron"],
    "antibiotique": ["Antibiotique"],
    "alambic":      ["Alambic"],
}

# Extraction
corpus = {}  # corpus[mot] = liste de phrases

for mot, articles in articles_par_mot.items():
    phrases_mot = []
    for titre in articles:
        page = wiki.page(titre)
        if page.exists():
            phrases_mot += extraire_phrases(page.text, mot)
        
    
    corpus[mot] = list(set(phrases_mot))  # dédoublonnage
    print(f"{mot} : {len(corpus[mot])} phrases extraites")

# Vérification
for mot, phrases in corpus.items():
    print(f"\n--- {mot.upper()} ({len(phrases)} phrases) ---")
    for p in phrases[:2]:
        print(f"  • {p[:100]}...")
    

In [ ]:
# Résumé du corpus
df_stats = pd.DataFrame({
    'mot': list(corpus.keys()),
    'n_phrases': [len(v) for v in corpus.values()],
    'polysemique': [m in mots_polysemique for m in corpus.keys()]
})
print(df_stats.sort_values('n_phrases'))